# EJAP publication figures — final design

Reproduces Figures 1–6 and Supplementary Figure S1 from frozen aggregate outputs. Exports vector PDF, editable SVG, and 600-dpi PNG.

**Important:** APRC is a mathematical pacing-regulation functional, not metabolic energy expenditure.

In [ ]:
from pathlib import Path
import os, sys

# When opened from GitHub in Colab, Colab already downloads this notebook only, not the full repository.
# We therefore read the frozen CSVs directly from raw.githubusercontent.com instead of cloning the repo.
IN_COLAB = 'google.colab' in sys.modules

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if IN_COLAB:
    BASE='https://raw.githubusercontent.com/ardominguezm/marathon-pacing-regulation/main/data/processed_results/'
    OUT=Path('/content/final_ejap_figures')
    OUT.mkdir(parents=True,exist_ok=True)
    def read_csv(name): return pd.read_csv(BASE+name)
else:
    ROOT=Path.cwd()
    if not (ROOT/'data'/'processed_results').exists() and (ROOT.parent/'data'/'processed_results').exists(): ROOT=ROOT.parent
    DATA=ROOT/'data'/'processed_results'
    OUT=ROOT/'figures'/'final_ejap'
    OUT.mkdir(parents=True,exist_ok=True)
    def read_csv(name): return pd.read_csv(DATA/name)

plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],'font.size':9,'axes.labelsize':9,'axes.titlesize':10,'xtick.labelsize':8,'ytick.labelsize':8,'legend.fontsize':8,'axes.spines.top':False,'axes.spines.right':False,'savefig.dpi':600,'savefig.bbox':'tight','pdf.fonttype':42,'ps.fonttype':42})
def save(fig,stem):
    fig.savefig(OUT/f'{stem}.pdf'); fig.savefig(OUT/f'{stem}.svg'); fig.savefig(OUT/f'{stem}.png',dpi=600); plt.show()
h=read_csv('information_horizon.csv'); rep=read_csv('horizon_city_year_replication.csv'); perf=read_csv('performance_horizon.csv'); sp=read_csv('spatial_prc_progression.csv'); cv=read_csv('aprc_vs_cv.csv'); dec=read_csv('decoupled_absolute_late_velocity.csv')

## Figure 1 — Conceptual measurement framework

In [ ]:
fig,ax=plt.subplots(figsize=(6.8,3.5)); x=np.array([10,15,20,25,30,35,40]); r=np.array([1.00,1.015,.995,.96,.92,.86,.83])
ax.plot(x,r,'o-',lw=1.8,ms=5); ax.axvspan(10,20,alpha=.08); ax.axvspan(30,40,alpha=.08); ax.axvline(20,ls='--',lw=1)
ax.text(15,1.03,'Observed regulation through horizon $x$',ha='center'); ax.text(20.5,.835,'information signal reproducible by ~20 km',fontsize=8); ax.text(35,.90,'late-race outcome\n30–40 km',ha='center')
ax.set(xlabel='Race distance (km)',ylabel='Normalized segment velocity  $r=v/v_{ref}$',title='From routine split times to a horizon-specific pacing-regulation functional')
ax.text(.02,.05,r'$\mathcal{R}(x)=m(x)^{-1}\sum [r(x_{j+1})-r(x_j)]^2$',transform=ax.transAxes); ax.text(.99,.02,'Schematic — not an individual runner',ha='right',transform=ax.transAxes,fontsize=7); save(fig,'Fig1_conceptual_framework')

## Figure 2 — Information horizon

In [ ]:
hp=h[h.Horizon_km.isin([15,20,25])].copy(); fig,ax=plt.subplots(figsize=(6.8,4.0)); ax.plot(hp.Horizon_km,hp.Delta_R2,'o-',lw=2,ms=6); ax.axhline(.005,ls=':',lw=1); ax.axvline(20,ls='--',lw=1)
for _,q in hp.iterrows(): ax.annotate(f'{q.Delta_R2:.4f}',(q.Horizon_km,q.Delta_R2),xytext=(0,7),textcoords='offset points',ha='center')
ax.text(20.4,max(hp.Delta_R2)*.72,'information horizon\ncriterion first met',va='center'); ax.set_xticks([15,20,25]); ax.set(xlabel='Observation horizon (km)',ylabel='Incremental explained variance  $\Delta R^2$',title='Information about subsequent late-race capacity emerges with distance'); save(fig,'Fig2_information_horizon')

## Figure 3 — Race-year replication

In [ ]:
rr=rep[rep.Horizon_km.isin([20,25])].copy(); rr['label']=rr.City.astype(str)+' '+rr.Year.astype(str); labels=sorted(rr.label.unique()); fig,ax=plt.subplots(figsize=(6.8,5.2))
for j,hor in enumerate([20,25]):
    d=rr[rr.Horizon_km==hor].set_index('label').loc[labels]; y=np.arange(len(labels))+(j-.5)*.18; ax.errorbar(d.beta,y,xerr=[d.beta-d.CI_low,d.CI_high-d.beta],fmt='o',ms=3.5,capsize=2,label=f'{hor} km')
ax.axvline(0,lw=1); ax.set_yticks(np.arange(len(labels)),labels); ax.invert_yaxis(); ax.set(xlabel='Standardized APRC coefficient (95% CI)',title='The association replicates across all 16 race-year cohorts'); ax.legend(frameon=False); save(fig,'Fig3_city_year_replication')

## Figure 4 — Spatial progression

In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.0)); x=np.arange(len(sp)); ax.errorbar(x,sp.Median,yerr=[sp.Median-sp.P25,sp.P75-sp.Median],fmt='o',capsize=5,lw=1.5,ms=6,label='Median (IQR)'); ax.plot(x,sp.Mean,'s--',lw=1.2,ms=5,label='Mean'); ax.set_xticks(x,sp.Region); ax.set(ylabel='Pacing regulation cost',xlabel='Race region',title='Local pacing-regulation cost increases across the marathon'); ax.legend(frameon=False); save(fig,'Fig4_spatial_progression')

## Figure 5 — Performance-stratified information curves

In [ ]:
fig,ax=plt.subplots(figsize=(6.8,4.2))
for name,d in perf[perf.Horizon_km.isin([15,20,25])].groupby('Performance'):
    d=d.sort_values('Horizon_km'); ax.plot(d.Horizon_km,d.Delta_R2,'o-',lw=1.5,ms=4,label=str(name))
ax.axvline(20,ls='--',lw=1); ax.set_xticks([15,20,25]); ax.set(xlabel='Observation horizon (km)',ylabel='Incremental explained variance  $\Delta R^2$',title='Horizon-dependent information persists across performance strata'); ax.legend(frameon=False,ncol=2); save(fig,'Fig5_performance_strata')

## Figure 6 — Confirmatory outcome de-coupling

In [ ]:
fig,ax=plt.subplots(figsize=(6.8,4.0)); ax.plot(dec.horizon_km,dec.Delta_R2,'o-',lw=2,ms=6); ax.axvline(20,ls='--',lw=1)
for _,q in dec.iterrows(): ax.annotate(f'{q.Delta_R2:.4f}',(q.horizon_km,q.Delta_R2),xytext=(0,7),textcoords='offset points',ha='center')
ax.set_xticks([15,20,25]); ax.set(xlabel='Observation horizon (km)',ylabel='Incremental explained variance  $\Delta R^2$',title='Information horizon persists with absolute late-race velocity'); ax.text(.02,.94,r'Outcome: mean 30–40-km velocity; baseline $R^2=0.806$',transform=ax.transAxes,va='top'); save(fig,'Fig6_decoupled_confirmation')

## Supplementary Figure S1 — APRC beyond CV

In [ ]:
d=cv[cv.Horizon_km.isin([15,20,25])]; fig,ax=plt.subplots(figsize=(6.2,4.0)); ax.plot(d.Horizon_km,d.Delta_R2_APRC_given_CV,'o-',lw=2,ms=6); ax.set_xticks([15,20,25]); ax.set(xlabel='Observation horizon (km)',ylabel=r'$\Delta R^2$ after adding APRC to CV model',title='Ordered local regulation adds information beyond global variability'); save(fig,'FigS1_APRC_beyond_CV')

## Output
In Colab, all files are written to `/content/final_ejap_figures/`. Locally, they are written to `figures/final_ejap/`.

Optional in Colab: run `!zip -r /content/final_ejap_figures.zip /content/final_ejap_figures` to download everything at once.